# 06 — Downstream Activity Detection Evaluation

This notebook implements the exact final evaluation design:

```text
1. train on 10 real subjects -> test on 3 real subjects
2. train on 10 real subjects -> test on 3 synthetic subjects
3. train on 10 synthetic subjects -> test on 3 real subjects
4. train on 10 real + 10 synthetic subjects -> test on 3 real subjects
```

This version keeps aeon MiniRocket/Rocket, but handles the low-variation error by **dropping only the problematic flat windows from the temporary aeon train/test arrays**.

It does **not** modify your saved real or synthetic data.

Default setting:

```text
framework: aeon MiniRocket/Rocket
modality: fused
low-variation strategy: drop_cases
```


In [1]:

# ============================================================
# 06_downstream_activity_evaluation_kovae.py
#
# Downstream human activity detection using real and synthetic KoVAE data.
#
# Adapted from friend's eval_new.ipynb, but adjusted to this project:
#
# Real data:
#   data/processed/native_rates/
#
# Synthetic data:
#   data/synthetic_subjects/kovae/rollout_v1/
#   data/synthetic_subjects/kovae/posterior_bank_v2/
#
# Results:
#   results/downstream/kovae/
#
# Figures:
#   figures/downstream/kovae/
#
# Models:
#   models/downstream/kovae/
#
# Default:
#   Framework: aeon MiniRocket/Rocket
#   Modality: fused multi-channel
#
# Optional:
#   Enable tsai if installed and if GPU/runtime is available.
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Tuple
import inspect
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_fscore_support,
)


# ============================================================
# Configuration
# ============================================================

EVAL_CONFIG = {
    "project_root": "/home/iailab42/khans1/projects/ir",

    "model_family": "kovae",

    "real_dir": "data/processed/native_rates",
    "synthetic_base_dir": "data/synthetic_subjects/kovae",

    "results_base_dir": "results/downstream",
    "figures_base_dir": "figures/downstream",
    "models_base_dir": "models/downstream",

    "synthetic_methods": ["rollout_v1", "posterior_bank_v2"],

    "method_display_names": {
        "rollout_v1": "KoVAE-Rollout",
        "posterior_bank_v2": "KoVAE-Posterior",
    },

    # Default compact final evaluation.
    # You can add "tsai" if installed.
    "frameworks": ["aeon"],

    # Options:
    #   "acc", "bvp", "eda", "temp", "fused"
    # Recommended final: ["fused"]
    "modalities": ["fused"],

    # Required by final evaluation design:
    # train on 10 real subjects -> test on 3 synthetic subjects
    "include_real_to_synthetic": True,

    # Synthetic test subjects used for real_to_synthetic.
    # The synthetic training experiments still use all 10 synthetic subjects.
    "synthetic_test_subject_selection": "first_n",
    "num_synthetic_test_subjects": 3,
    "specific_synthetic_test_subjects": [],

    # Split used throughout this project.
    "train_subjects": ["S1", "S2", "S3", "S4", "S5", "S6", "S9", "S11", "S12", "S13"],
    "val_subjects": ["S14", "S15"],
    "test_subjects": ["S7", "S8", "S10"],

    "activity_ids": [1, 2, 3, 4, 5, 6, 7, 8],

    # Fused view resamples all modalities to BVP length.
    "fused_target_len": 512,

    # If runtime is too slow, set this to e.g. 30000 or 20000.
    # None means use all available training windows.
    "max_train_windows_per_experiment": None,

    # aeon settings.
    "aeon_n_kernels": 5000,
    "aeon_n_jobs": -1,

    # MiniRocket/Rocket cannot handle case/channel pairs with almost zero std.
    # This only repairs the temporary classifier input, not the saved real/synthetic data.
    "aeon_fix_low_variation": True,
    "aeon_low_variation_strategy": "drop_cases",
    "aeon_min_std": 1e-6,
    "aeon_ramp_scale": 1e-3,

    # tsai settings.
    "tsai_arch": "InceptionTimePlus",
    "tsai_epochs": 10,
    "tsai_batch_size": 384,
    "tsai_lr": 1e-3,

    "random_seed": 42,

    "save_models": True,
    "save_confusion_matrix_figures": True,
    "show_confusion_matrix_figures": False,
}


# ============================================================
# Native data configuration
# ============================================================

RAW_ARRAY_CONFIGS = {
    "acc": {
        "real_filename": "all_X_acc_32hz.npy",
        "syn_filename": "generated_subjects_X_acc_32hz.npy",
        "expected_shape_tail": (256, 3),
        "native_hz": 32,
        "channel_names": ["ACC_x", "ACC_y", "ACC_z"],
    },
    "bvp": {
        "real_filename": "all_X_bvp_64hz.npy",
        "syn_filename": "generated_subjects_X_bvp_64hz.npy",
        "expected_shape_tail": (512, 1),
        "native_hz": 64,
        "channel_names": ["BVP"],
    },
    "slow": {
        "real_filename": "all_X_slow_4hz.npy",
        "syn_filename": "generated_subjects_X_slow_4hz.npy",
        "expected_shape_tail": (32, 2),
        "native_hz": 4,
        "channel_names": ["EDA", "TEMP"],
    },
}

FUSED_CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]


# ============================================================
# Basic helpers
# ============================================================

def set_random_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def require_file(path: Path) -> Path:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return path


def make_dirs(*dirs: Path) -> None:
    for directory in dirs:
        directory.mkdir(parents=True, exist_ok=True)


def save_json(data: Dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def safe_name(value: str) -> str:
    value = str(value)
    for ch in [" ", "/", "\\", "|", ":", ";", ",", "(", ")", "[", "]", "{", "}"]:
        value = value.replace(ch, "_")
    while "__" in value:
        value = value.replace("__", "_")
    return value.strip("_")


def subject_sort_key(value):
    text = str(value)
    if text.startswith("S") and text[1:].isdigit():
        return (0, int(text[1:]))
    digits = "".join(ch for ch in text if ch.isdigit())
    if digits:
        return (1, int(digits), text)
    return (2, text)


def method_display_name(method_name: str, config: Dict) -> str:
    return config.get("method_display_names", {}).get(method_name, method_name)


def get_base_paths(config: Dict) -> Dict[str, Path]:
    root = Path(config["project_root"])
    model_family = str(config["model_family"])

    return {
        "root": root,
        "real_dir": root / config["real_dir"],
        "synthetic_base_dir": root / config["synthetic_base_dir"],
        "results_model_dir": root / config["results_base_dir"] / model_family,
        "figures_model_dir": root / config["figures_base_dir"] / model_family,
        "models_model_dir": root / config["models_base_dir"] / model_family,
        "configs_dir": root / "configs",
    }


def get_run_dirs(base_paths: Dict[str, Path], framework: str, experiment_name: str, modality: str) -> Dict[str, Path]:
    key = f"{safe_name(framework)}__{safe_name(experiment_name)}__{safe_name(modality)}"

    return {
        "run_key": key,
        "results_dir": base_paths["results_model_dir"] / "runs" / key,
        "figures_dir": base_paths["figures_model_dir"] / "confusion_matrices",
        "models_dir": base_paths["models_model_dir"] / framework,
        "predictions_dir": base_paths["results_model_dir"] / "predictions",
        "reports_dir": base_paths["results_model_dir"] / "per_activity_reports",
        "confusion_csv_dir": base_paths["results_model_dir"] / "confusion_matrices",
    }


def labels_to_indices(y: np.ndarray, activity_ids: List[int]) -> np.ndarray:
    mapping = {int(label): i for i, label in enumerate(activity_ids)}
    y = np.asarray(y, dtype=np.int64)

    out = np.empty_like(y, dtype=np.int64)
    for i, label in enumerate(y):
        label = int(label)
        if label not in mapping:
            raise ValueError(f"Unexpected label {label}; expected {activity_ids}")
        out[i] = mapping[label]

    return out


def indices_to_labels(y_idx: np.ndarray, activity_ids: List[int]) -> np.ndarray:
    activity_ids = [int(x) for x in activity_ids]
    return np.asarray([activity_ids[int(i)] for i in y_idx], dtype=np.int64)


def to_channels_first(X_native: np.ndarray) -> np.ndarray:
    X_native = np.asarray(X_native, dtype=np.float32)

    if X_native.ndim != 3:
        raise ValueError(f"Expected [N,T,C], got {X_native.shape}")

    return np.transpose(X_native, (0, 2, 1)).astype(np.float32)


def fix_low_variation_for_aeon(
    X_channels_first: np.ndarray,
    min_std: float = 1e-6,
    ramp_scale: float = 1e-5,
    verbose: bool = True,
) -> np.ndarray:
    """Repair nearly constant case/channel pairs for aeon MiniRocket/Rocket.

    aeon raises an error when any individual case/channel pair has std <= threshold.
    We add a very tiny deterministic zero-mean ramp only to those problematic pairs.
    This does not change saved data; it only changes the temporary classifier input.
    """
    X = np.asarray(X_channels_first, dtype=np.float32).copy()

    if X.ndim != 3:
        raise ValueError(f"Expected [N,C,T], got {X.shape}")

    std = X.std(axis=2)
    bad_mask = std <= float(min_std)
    num_bad = int(bad_mask.sum())

    if num_bad == 0:
        return X

    n_time = X.shape[2]
    ramp = np.linspace(-1.0, 1.0, n_time, dtype=np.float32)
    ramp = ramp - ramp.mean()
    ramp_std = float(ramp.std())

    if ramp_std == 0:
        raise ValueError("Internal ramp has zero std.")

    ramp = ramp / ramp_std
    ramp = ramp * float(ramp_scale)

    bad_cases, bad_channels = np.where(bad_mask)

    for case_idx, channel_idx in zip(bad_cases, bad_channels):
        X[case_idx, channel_idx, :] = X[case_idx, channel_idx, :] + ramp

    if verbose:
        print(
            f"aeon low-variation repair: fixed {num_bad} case/channel pair(s) "
            f"with std <= {min_std} using ramp_scale={ramp_scale}"
        )

    return X


def keep_nonflat_cases_for_aeon(
    X_channels_first: np.ndarray,
    y: np.ndarray,
    min_std: float = 1e-6,
    verbose: bool = True,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Drop windows where at least one channel is nearly constant.

    This is used only for aeon MiniRocket/Rocket because aeon rejects
    case/channel pairs with extremely low variation. The saved data is not changed.
    """
    X = np.asarray(X_channels_first, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)

    if X.ndim != 3:
        raise ValueError(f"Expected [N,C,T], got {X.shape}")

    std = X.std(axis=2)
    keep_mask = np.all(std > float(min_std), axis=1)

    kept = int(keep_mask.sum())
    dropped = int(len(keep_mask) - kept)

    if kept == 0:
        raise ValueError(
            "All cases would be dropped by aeon low-variation filtering. "
            "Use a lower aeon_min_std or use sklearn."
        )

    if verbose and dropped > 0:
        print(
            f"aeon low-variation filter: dropped {dropped} of {len(keep_mask)} windows "
            f"where at least one channel had std <= {min_std}"
        )

    return X[keep_mask], y[keep_mask], keep_mask


def resample_time_axis(X_native: np.ndarray, target_len: int) -> np.ndarray:
    X_native = np.asarray(X_native, dtype=np.float32)

    if X_native.ndim != 3:
        raise ValueError(f"Expected [N,T,C], got {X_native.shape}")

    n, old_len, channels = X_native.shape
    target_len = int(target_len)

    if old_len == target_len:
        return X_native.copy().astype(np.float32)

    if old_len < 2:
        raise ValueError(f"Cannot resample time axis with old_len={old_len}")

    old_positions = np.linspace(0.0, old_len - 1, target_len, dtype=np.float32)
    left = np.floor(old_positions).astype(np.int64)
    right = np.minimum(left + 1, old_len - 1)
    weight = (old_positions - left).astype(np.float32)

    out = (
        (1.0 - weight)[None, :, None] * X_native[:, left, :]
        + weight[None, :, None] * X_native[:, right, :]
    )

    return out.astype(np.float32)


def split_slow_to_eda_temp(X_slow: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X_slow = np.asarray(X_slow, dtype=np.float32)

    if X_slow.ndim != 3 or X_slow.shape[1:] != (32, 2):
        raise ValueError(f"Expected SLOW [N,32,2], got {X_slow.shape}")

    X_eda = X_slow[:, :, 0:1].astype(np.float32)
    X_temp = X_slow[:, :, 1:2].astype(np.float32)

    return X_eda, X_temp


def build_fused_native_view(X_view_dict: Dict[str, np.ndarray], target_len: int) -> np.ndarray:
    X_acc = resample_time_axis(X_view_dict["acc"], target_len)
    X_bvp = resample_time_axis(X_view_dict["bvp"], target_len)
    X_eda = resample_time_axis(X_view_dict["eda"], target_len)
    X_temp = resample_time_axis(X_view_dict["temp"], target_len)

    lengths = {
        "acc": len(X_acc),
        "bvp": len(X_bvp),
        "eda": len(X_eda),
        "temp": len(X_temp),
    }

    if len(set(lengths.values())) != 1:
        raise ValueError(f"Fused view length mismatch: {lengths}")

    return np.concatenate([X_acc, X_bvp, X_eda, X_temp], axis=2).astype(np.float32)


def stratified_subsample(
    X: np.ndarray,
    y: np.ndarray,
    max_samples: Optional[int],
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    if max_samples is None:
        return X, y

    max_samples = int(max_samples)

    if len(y) <= max_samples:
        return X, y

    rng = np.random.default_rng(seed)
    y = np.asarray(y, dtype=np.int64)

    selected_indices = []
    labels = np.unique(y)

    for label in labels:
        idx = np.where(y == label)[0]
        n_label = len(idx)
        n_take = max(1, int(round(max_samples * n_label / len(y))))
        n_take = min(n_take, n_label)
        selected_indices.append(rng.choice(idx, size=n_take, replace=False))

    selected_indices = np.concatenate(selected_indices)

    if len(selected_indices) > max_samples:
        selected_indices = rng.choice(selected_indices, size=max_samples, replace=False)

    rng.shuffle(selected_indices)

    return X[selected_indices], y[selected_indices]


def safe_display(df: pd.DataFrame, max_rows: int = 20) -> None:
    try:
        display(df)
    except Exception:
        print(df.head(max_rows).to_string(index=False))


# ============================================================
# Loading
# ============================================================

def check_raw_array_shape(X: np.ndarray, cfg: Dict, name: str) -> None:
    expected_tail = tuple(cfg["expected_shape_tail"])

    if X.ndim != 3 or tuple(X.shape[1:]) != expected_tail:
        raise ValueError(f"{name}: expected [N,{expected_tail[0]},{expected_tail[1]}], got {X.shape}")


def load_real_native_data(real_dir: Path, config: Dict) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    y = np.load(require_file(real_dir / "all_y.npy")).astype(np.int64)
    subjects = np.load(require_file(real_dir / "all_subject.npy"), allow_pickle=True).astype(str)

    if len(y) != len(subjects):
        raise ValueError(f"Real y/subject mismatch: {len(y)} vs {len(subjects)}")

    X = {}

    for key, cfg in RAW_ARRAY_CONFIGS.items():
        path = require_file(real_dir / cfg["real_filename"])
        arr = np.load(path).astype(np.float32)
        check_raw_array_shape(arr, cfg, f"Real {key}")

        if len(arr) != len(y):
            raise ValueError(f"Real {key}/y mismatch: {len(arr)} vs {len(y)}")

        X[key] = arr
        print(f"Loaded real {key}: {arr.shape}")

    X, y, subjects = filter_valid_activities(X, y, subjects, config["activity_ids"])

    return X, y, subjects


def load_synthetic_native_data(synthetic_dir: Path, config: Dict) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    y = np.load(require_file(synthetic_dir / "generated_subjects_all_y.npy")).astype(np.int64)
    subjects = np.load(
        require_file(synthetic_dir / "generated_subjects_all_subject.npy"),
        allow_pickle=True,
    ).astype(str)

    if len(y) != len(subjects):
        raise ValueError(f"Synthetic y/subject mismatch: {len(y)} vs {len(subjects)}")

    X = {}

    for key, cfg in RAW_ARRAY_CONFIGS.items():
        path = require_file(synthetic_dir / cfg["syn_filename"])
        arr = np.load(path).astype(np.float32)
        check_raw_array_shape(arr, cfg, f"Synthetic {key}")

        if len(arr) != len(y):
            raise ValueError(f"Synthetic {key}/y mismatch: {len(arr)} vs {len(y)}")

        X[key] = arr
        print(f"Loaded synthetic {key}: {arr.shape}")

    X, y, subjects = filter_valid_activities(X, y, subjects, config["activity_ids"])

    return X, y, subjects


def filter_valid_activities(
    X_dict: Dict[str, np.ndarray],
    y: np.ndarray,
    subjects: np.ndarray,
    activity_ids: List[int],
) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    keep = np.isin(y, np.asarray(activity_ids, dtype=np.int64))

    X_out = {key: value[keep].astype(np.float32) for key, value in X_dict.items()}
    y_out = y[keep].astype(np.int64)
    subjects_out = subjects[keep].astype(str)

    return X_out, y_out, subjects_out


def filter_by_subjects(
    X_dict: Dict[str, np.ndarray],
    y: np.ndarray,
    subjects: np.ndarray,
    selected_subjects: List[str],
) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    selected = set(str(s) for s in selected_subjects)
    keep = np.asarray([str(s) in selected for s in subjects], dtype=bool)

    X_out = {key: value[keep].astype(np.float32) for key, value in X_dict.items()}
    y_out = y[keep].astype(np.int64)
    subjects_out = subjects[keep].astype(str)

    return X_out, y_out, subjects_out


def select_synthetic_test_subjects(subjects: np.ndarray, config: Dict) -> List[str]:
    available = sorted(np.unique(subjects.astype(str)).tolist(), key=subject_sort_key)
    mode = str(config["synthetic_test_subject_selection"])

    if mode == "first_n":
        n = int(config["num_synthetic_test_subjects"])
        selected = available[:n]

    elif mode == "specific":
        selected = [str(x) for x in config["specific_synthetic_test_subjects"]]
        missing = sorted(set(selected) - set(available), key=subject_sort_key)

        if missing:
            raise ValueError(f"Requested synthetic test subjects not found: {missing}")

    else:
        raise ValueError("synthetic_test_subject_selection must be 'first_n' or 'specific'.")

    if len(selected) == 0:
        raise ValueError("No synthetic test subjects selected.")

    return selected


def validate_subject_split(subjects: np.ndarray, config: Dict) -> None:
    available = set(subjects.astype(str))

    missing_train = sorted(set(config["train_subjects"]) - available, key=subject_sort_key)
    missing_val = sorted(set(config["val_subjects"]) - available, key=subject_sort_key)
    missing_test = sorted(set(config["test_subjects"]) - available, key=subject_sort_key)

    if missing_train or missing_val or missing_test:
        raise ValueError(
            "Missing split subjects.\n"
            f"Missing train: {missing_train}\n"
            f"Missing val:   {missing_val}\n"
            f"Missing test:  {missing_test}"
        )

    if set(config["train_subjects"]) & set(config["val_subjects"]):
        raise ValueError("Train/val overlap.")

    if set(config["train_subjects"]) & set(config["test_subjects"]):
        raise ValueError("Train/test overlap.")

    if set(config["val_subjects"]) & set(config["test_subjects"]):
        raise ValueError("Val/test overlap.")

    print("Subject split OK.")


def prepare_real_splits(
    real_X_all: Dict[str, np.ndarray],
    real_y_all: np.ndarray,
    real_subjects_all: np.ndarray,
    config: Dict,
) -> Dict[str, Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]]:
    validate_subject_split(real_subjects_all, config)

    return {
        "train": filter_by_subjects(
            real_X_all,
            real_y_all,
            real_subjects_all,
            config["train_subjects"],
        ),
        "val": filter_by_subjects(
            real_X_all,
            real_y_all,
            real_subjects_all,
            config["val_subjects"],
        ),
        "test": filter_by_subjects(
            real_X_all,
            real_y_all,
            real_subjects_all,
            config["test_subjects"],
        ),
    }


def build_eval_raw_views(X_raw: Dict[str, np.ndarray], config: Dict) -> Dict[str, np.ndarray]:
    X_eda, X_temp = split_slow_to_eda_temp(X_raw["slow"])

    views = {
        "acc": X_raw["acc"],
        "bvp": X_raw["bvp"],
        "eda": X_eda,
        "temp": X_temp,
    }

    if "fused" in config["modalities"]:
        views["fused"] = build_fused_native_view(
            X_view_dict=views,
            target_len=int(config["fused_target_len"]),
        )

    return views


def get_modality_info(modality: str, config: Dict) -> Dict:
    if modality == "acc":
        return {
            "native_hz": 32,
            "channel_names": ["ACC_x", "ACC_y", "ACC_z"],
            "description": "ACC 32 Hz native view",
        }

    if modality == "bvp":
        return {
            "native_hz": 64,
            "channel_names": ["BVP"],
            "description": "BVP 64 Hz native view",
        }

    if modality == "eda":
        return {
            "native_hz": 4,
            "channel_names": ["EDA"],
            "description": "EDA 4 Hz native view",
        }

    if modality == "temp":
        return {
            "native_hz": 4,
            "channel_names": ["TEMP"],
            "description": "TEMP 4 Hz native view",
        }

    if modality == "fused":
        return {
            "native_hz": 64,
            "channel_names": FUSED_CHANNEL_NAMES,
            "description": "Fused ACC+BVP+EDA+TEMP resampled to length 512",
        }

    raise ValueError(f"Unknown modality: {modality}")


def save_split_summary(
    real_splits: Dict[str, Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]],
    synthetic_summaries: Dict[str, Tuple[np.ndarray, np.ndarray]],
    synthetic_test_summaries: Dict[str, Tuple[np.ndarray, np.ndarray]],
    base_paths: Dict[str, Path],
    config: Dict,
) -> pd.DataFrame:
    rows = []

    for split_name, (_, y, subjects) in real_splits.items():
        row = {
            "dataset": f"real_{split_name}",
            "num_windows": int(len(y)),
            "subjects": ",".join(sorted(np.unique(subjects.astype(str)), key=subject_sort_key)),
        }

        for activity in config["activity_ids"]:
            row[f"activity_{activity}_windows"] = int(np.sum(y == int(activity)))

        rows.append(row)

    for method_name, (y, subjects) in synthetic_summaries.items():
        row = {
            "dataset": f"synthetic_train_all10_{method_name}",
            "num_windows": int(len(y)),
            "subjects": ",".join(sorted(np.unique(subjects.astype(str)), key=subject_sort_key)),
        }

        for activity in config["activity_ids"]:
            row[f"activity_{activity}_windows"] = int(np.sum(y == int(activity)))

        rows.append(row)

    for method_name, (y, subjects) in synthetic_test_summaries.items():
        row = {
            "dataset": f"synthetic_test_3subjects_{method_name}",
            "num_windows": int(len(y)),
            "subjects": ",".join(sorted(np.unique(subjects.astype(str)), key=subject_sort_key)),
        }

        for activity in config["activity_ids"]:
            row[f"activity_{activity}_windows"] = int(np.sum(y == int(activity)))

        rows.append(row)

    df = pd.DataFrame(rows)

    out_path = base_paths["results_model_dir"] / "data_split_summary.csv"
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return df


# ============================================================
# Metrics and saving
# ============================================================

def compute_classification_metrics(y_true: np.ndarray, y_pred: np.ndarray, activity_ids: List[int]) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(
            precision_score(y_true, y_pred, labels=activity_ids, average="macro", zero_division=0)
        ),
        "macro_recall": float(
            recall_score(y_true, y_pred, labels=activity_ids, average="macro", zero_division=0)
        ),
        "macro_f1": float(
            f1_score(y_true, y_pred, labels=activity_ids, average="macro", zero_division=0)
        ),
        "weighted_precision": float(
            precision_score(y_true, y_pred, labels=activity_ids, average="weighted", zero_division=0)
        ),
        "weighted_recall": float(
            recall_score(y_true, y_pred, labels=activity_ids, average="weighted", zero_division=0)
        ),
        "weighted_f1": float(
            f1_score(y_true, y_pred, labels=activity_ids, average="weighted", zero_division=0)
        ),
    }

    metrics["balanced_accuracy"] = metrics["macro_recall"]

    return metrics


def compute_per_activity_metrics(y_true: np.ndarray, y_pred: np.ndarray, activity_ids: List[int]) -> pd.DataFrame:
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=activity_ids,
        zero_division=0,
    )

    rows = []

    for i, activity in enumerate(activity_ids):
        activity = int(activity)
        true_mask = y_true == activity
        correct = int(np.sum(true_mask & (y_pred == activity)))
        total = int(np.sum(true_mask))
        activity_accuracy = float(correct / total) if total > 0 else np.nan

        rows.append(
            {
                "activity_label": activity,
                "precision": float(precision[i]),
                "recall": float(recall[i]),
                "f1": float(f1[i]),
                "support": int(support[i]),
                "correct_true_activity_windows": correct,
                "total_true_activity_windows": total,
                "true_activity_window_accuracy": activity_accuracy,
            }
        )

    return pd.DataFrame(rows)


def plot_and_save_confusion_matrix(
    cm: np.ndarray,
    labels: List[int],
    title: str,
    save_path: Path,
    config: Dict,
) -> None:
    if not bool(config["save_confusion_matrix_figures"]):
        return

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted activity")
    ax.set_ylabel("True activity")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels([str(x) for x in labels])
    ax.set_yticklabels([str(x) for x in labels])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = int(cm[i, j])
            if value > 0:
                ax.text(j, i, str(value), ha="center", va="center", fontsize=8)

    fig.colorbar(im, ax=ax)
    fig.tight_layout()

    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    print("Saved:", save_path)

    if bool(config["show_confusion_matrix_figures"]):
        plt.show()

    plt.close(fig)


def save_run_outputs(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    framework: str,
    model_name: str,
    experiment_name: str,
    synthetic_method: str,
    modality: str,
    modality_info: Dict,
    base_paths: Dict[str, Path],
    config: Dict,
) -> pd.DataFrame:
    run_dirs = get_run_dirs(base_paths, framework, experiment_name, modality)
    make_dirs(
        run_dirs["results_dir"],
        run_dirs["predictions_dir"],
        run_dirs["reports_dir"],
        run_dirs["confusion_csv_dir"],
        run_dirs["figures_dir"],
    )

    key = run_dirs["run_key"]

    pred_df = pd.DataFrame(
        {
            "y_true": np.asarray(y_true, dtype=np.int64),
            "y_pred": np.asarray(y_pred, dtype=np.int64),
            "correct": np.asarray(y_true, dtype=np.int64) == np.asarray(y_pred, dtype=np.int64),
        }
    )

    pred_path = run_dirs["predictions_dir"] / f"{key}_predictions.csv"
    pred_df.to_csv(pred_path, index=False)
    print("Saved:", pred_path)

    cm = confusion_matrix(y_true, y_pred, labels=config["activity_ids"])

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{a}" for a in config["activity_ids"]],
        columns=[f"pred_{a}" for a in config["activity_ids"]],
    )

    cm_path = run_dirs["confusion_csv_dir"] / f"{key}_confusion_matrix.csv"
    cm_df.to_csv(cm_path)
    print("Saved:", cm_path)

    cm_fig_path = run_dirs["figures_dir"] / f"{key}_confusion_matrix.png"
    plot_and_save_confusion_matrix(
        cm=cm,
        labels=config["activity_ids"],
        title=f"{framework} | {experiment_name} | {modality}",
        save_path=cm_fig_path,
        config=config,
    )

    activity_df = compute_per_activity_metrics(
        y_true=np.asarray(y_true, dtype=np.int64),
        y_pred=np.asarray(y_pred, dtype=np.int64),
        activity_ids=config["activity_ids"],
    )

    activity_df.insert(0, "framework", framework)
    activity_df.insert(1, "model", model_name)
    activity_df.insert(2, "experiment", experiment_name)
    activity_df.insert(3, "synthetic_method", synthetic_method)
    activity_df.insert(4, "modality", modality)
    activity_df.insert(5, "native_hz", int(modality_info["native_hz"]))
    activity_df.insert(6, "channels", ",".join(modality_info["channel_names"]))

    activity_path = run_dirs["reports_dir"] / f"{key}_per_activity_metrics.csv"
    activity_df.to_csv(activity_path, index=False)
    print("Saved:", activity_path)

    return activity_df


# ============================================================
# Experiments
# ============================================================

def build_experiment(
    name: str,
    synthetic_method: str,
    train_X: np.ndarray,
    train_y: np.ndarray,
    val_X: np.ndarray,
    val_y: np.ndarray,
    test_X: np.ndarray,
    test_y: np.ndarray,
    modality: str,
    config: Dict,
) -> Dict:
    train_X, train_y = stratified_subsample(
        train_X,
        train_y,
        max_samples=config["max_train_windows_per_experiment"],
        seed=int(config["random_seed"]) + sum(ord(ch) for ch in str(name)) % 10000,
    )

    return {
        "name": name,
        "synthetic_method": synthetic_method,
        "modality": modality,
        "train_X": train_X,
        "train_y": train_y,
        "val_X": val_X,
        "val_y": val_y,
        "test_X": test_X,
        "test_y": test_y,
    }


def build_experiments_for_modality(
    modality: str,
    real_train_views: Dict[str, np.ndarray],
    real_val_views: Dict[str, np.ndarray],
    real_test_views: Dict[str, np.ndarray],
    real_train_y: np.ndarray,
    real_val_y: np.ndarray,
    real_test_y: np.ndarray,
    synthetic_views_by_method: Dict[str, Dict[str, np.ndarray]],
    synthetic_y_by_method: Dict[str, np.ndarray],
    synthetic_test_views_by_method: Dict[str, Dict[str, np.ndarray]],
    synthetic_test_y_by_method: Dict[str, np.ndarray],
    config: Dict,
) -> List[Dict]:
    experiments = []

    experiments.append(
        build_experiment(
            name="real_to_real",
            synthetic_method="none",
            train_X=real_train_views[modality],
            train_y=real_train_y,
            val_X=real_val_views[modality],
            val_y=real_val_y,
            test_X=real_test_views[modality],
            test_y=real_test_y,
            modality=modality,
            config=config,
        )
    )

    for method_name in config["synthetic_methods"]:
        syn_views = synthetic_views_by_method[method_name]
        syn_y = synthetic_y_by_method[method_name]

        experiments.append(
            build_experiment(
                name=f"synthetic_to_real__{method_name}",
                synthetic_method=method_name,
                train_X=syn_views[modality],
                train_y=syn_y,
                val_X=real_val_views[modality],
                val_y=real_val_y,
                test_X=real_test_views[modality],
                test_y=real_test_y,
                modality=modality,
                config=config,
            )
        )

        experiments.append(
            build_experiment(
                name=f"real_plus_synthetic_to_real__{method_name}",
                synthetic_method=method_name,
                train_X=np.concatenate([real_train_views[modality], syn_views[modality]], axis=0),
                train_y=np.concatenate([real_train_y, syn_y], axis=0),
                val_X=real_val_views[modality],
                val_y=real_val_y,
                test_X=real_test_views[modality],
                test_y=real_test_y,
                modality=modality,
                config=config,
            )
        )

        if bool(config["include_real_to_synthetic"]):
            syn_test_views = synthetic_test_views_by_method[method_name]
            syn_test_y = synthetic_test_y_by_method[method_name]

            experiments.append(
                build_experiment(
                    name=f"real_to_3synthetic__{method_name}",
                    synthetic_method=method_name,
                    train_X=real_train_views[modality],
                    train_y=real_train_y,
                    val_X=real_val_views[modality],
                    val_y=real_val_y,
                    test_X=syn_test_views[modality],
                    test_y=syn_test_y,
                    modality=modality,
                    config=config,
                )
            )

    return experiments


def save_shape_report(experiments_by_modality: Dict[str, List[Dict]], base_paths: Dict[str, Path]) -> pd.DataFrame:
    rows = []

    for modality, experiments in experiments_by_modality.items():
        for exp in experiments:
            rows.append(
                {
                    "experiment": exp["name"],
                    "synthetic_method": exp["synthetic_method"],
                    "modality": modality,
                    "train_shape_native_N_T_C": list(exp["train_X"].shape),
                    "val_shape_native_N_T_C": list(exp["val_X"].shape),
                    "test_shape_native_N_T_C": list(exp["test_X"].shape),
                    "train_windows": int(len(exp["train_y"])),
                    "val_windows": int(len(exp["val_y"])),
                    "test_windows": int(len(exp["test_y"])),
                }
            )

    df = pd.DataFrame(rows)

    out_path = base_paths["results_model_dir"] / "downstream_shape_report.csv"
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return df


# ============================================================
# aeon
# ============================================================

def import_aeon_classifier_class():
    try:
        from aeon.classification.convolution_based import MiniRocketClassifier
        return MiniRocketClassifier
    except Exception:
        try:
            from aeon.classification.convolution_based import RocketClassifier
            return RocketClassifier
        except Exception as exc:
            raise ImportError(f"Could not import MiniRocketClassifier or RocketClassifier: {repr(exc)}")


def make_aeon_classifier(config: Dict):
    Classifier = import_aeon_classifier_class()

    params = inspect.signature(Classifier).parameters
    kwargs = {}

    if "n_kernels" in params:
        kwargs["n_kernels"] = int(config["aeon_n_kernels"])

    if "num_kernels" in params:
        kwargs["num_kernels"] = int(config["aeon_n_kernels"])

    if "n_jobs" in params:
        kwargs["n_jobs"] = int(config["aeon_n_jobs"])

    if "random_state" in params:
        kwargs["random_state"] = int(config["random_seed"])

    return Classifier(**kwargs)


def run_aeon_experiment(
    exp: Dict,
    modality_info: Dict,
    base_paths: Dict[str, Path],
    config: Dict,
) -> Tuple[Dict, pd.DataFrame]:
    try:
        clf = make_aeon_classifier(config)
    except Exception as exc:
        raise ImportError(f"aeon is unavailable or classifier could not be created: {repr(exc)}")

    X_train = to_channels_first(exp["train_X"])
    X_test = to_channels_first(exp["test_X"])
    y_train = exp["train_y"].astype(np.int64)
    y_test = exp["test_y"].astype(np.int64)

    train_keep_mask = np.ones(len(y_train), dtype=bool)
    test_keep_mask = np.ones(len(y_test), dtype=bool)

    if bool(config.get("aeon_fix_low_variation", True)):
        strategy = str(config.get("aeon_low_variation_strategy", "drop_cases"))

        if strategy == "drop_cases":
            X_train, y_train, train_keep_mask = keep_nonflat_cases_for_aeon(
                X_train,
                y_train,
                min_std=float(config.get("aeon_min_std", 1e-6)),
                verbose=True,
            )
            X_test, y_test, test_keep_mask = keep_nonflat_cases_for_aeon(
                X_test,
                y_test,
                min_std=float(config.get("aeon_min_std", 1e-6)),
                verbose=True,
            )

        elif strategy == "ramp":
            X_train = fix_low_variation_for_aeon(
                X_train,
                min_std=float(config.get("aeon_min_std", 1e-6)),
                ramp_scale=float(config.get("aeon_ramp_scale", 1e-3)),
                verbose=True,
            )
            X_test = fix_low_variation_for_aeon(
                X_test,
                min_std=float(config.get("aeon_min_std", 1e-6)),
                ramp_scale=float(config.get("aeon_ramp_scale", 1e-3)),
                verbose=True,
            )

        else:
            raise ValueError("aeon_low_variation_strategy must be 'drop_cases' or 'ramp'.")

    print("\n" + "=" * 90)
    print(f"[aeon] {exp['name']} | modality={exp['modality']}")
    print("=" * 90)
    print("Model:", clf.__class__.__name__)
    print("Train:", X_train.shape, y_train.shape)
    print("Test: ", X_test.shape, y_test.shape)

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test).astype(np.int64)

    metrics = compute_classification_metrics(y_test, y_pred, config["activity_ids"])

    activity_df = save_run_outputs(
        y_true=y_test,
        y_pred=y_pred,
        framework="aeon",
        model_name=clf.__class__.__name__,
        experiment_name=exp["name"],
        synthetic_method=exp["synthetic_method"],
        modality=exp["modality"],
        modality_info=modality_info,
        base_paths=base_paths,
        config=config,
    )

    if bool(config["save_models"]):
        try:
            import joblib
            run_dirs = get_run_dirs(base_paths, "aeon", exp["name"], exp["modality"])
            make_dirs(run_dirs["models_dir"])
            model_path = run_dirs["models_dir"] / f"{run_dirs['run_key']}.joblib"
            joblib.dump(clf, model_path)
            print("Saved:", model_path)
        except Exception as exc:
            warnings.warn(f"Could not save aeon model: {repr(exc)}")

    row = {
        "framework": "aeon",
        "model": clf.__class__.__name__,
        "experiment": exp["name"],
        "synthetic_method": exp["synthetic_method"],
        "synthetic_method_display_name": (
            "none" if exp["synthetic_method"] == "none"
            else method_display_name(exp["synthetic_method"], config)
        ),
        "modality": exp["modality"],
        "native_hz": int(modality_info["native_hz"]),
        "channels": ",".join(modality_info["channel_names"]),
        "description": modality_info["description"],
        "train_shape_native_N_T_C": list(exp["train_X"].shape),
        "test_shape_native_N_T_C": list(exp["test_X"].shape),
        "train_shape_framework_N_C_T": list(X_train.shape),
        "test_shape_framework_N_C_T": list(X_test.shape),
        "train_windows": int(len(y_train)),
        "test_windows": int(len(y_test)),
        "train_windows_original_before_aeon_filter": int(len(exp["train_y"])),
        "test_windows_original_before_aeon_filter": int(len(exp["test_y"])),
        "train_windows_dropped_by_aeon_filter": int(len(exp["train_y"]) - len(y_train)),
        "test_windows_dropped_by_aeon_filter": int(len(exp["test_y"]) - len(y_test)),
        "aeon_n_kernels": int(config["aeon_n_kernels"]),
        "aeon_n_jobs": int(config["aeon_n_jobs"]),
        "aeon_fix_low_variation": bool(config.get("aeon_fix_low_variation", True)),
        "aeon_low_variation_strategy": str(config.get("aeon_low_variation_strategy", "drop_cases")),
        "aeon_min_std": float(config.get("aeon_min_std", 1e-6)),
        "aeon_ramp_scale": float(config.get("aeon_ramp_scale", 1e-3)),
        **metrics,
    }

    print(json.dumps(row, indent=2))

    return row, activity_df


# ============================================================
# tsai
# ============================================================

def import_tsai_stuff(config: Dict):
    try:
        import torch
        import tsai.all as tsai_all

        from tsai.all import (
            get_ts_dls,
            ts_learner,
            TSClassification,
            TSStandardize,
        )

        try:
            from fastai.metrics import accuracy
        except Exception:
            from tsai.all import accuracy

    except Exception as exc:
        raise ImportError(f"Could not import tsai/fastai: {repr(exc)}")

    arch_name = str(config["tsai_arch"])

    if not hasattr(tsai_all, arch_name):
        common_options = [
            name for name in ["InceptionTimePlus", "InceptionTime", "ResNet", "FCN", "TST"]
            if hasattr(tsai_all, name)
        ]
        raise ValueError(
            f"tsai architecture '{arch_name}' was not found. Common available options: {common_options}"
        )

    return {
        "torch": torch,
        "get_ts_dls": get_ts_dls,
        "ts_learner": ts_learner,
        "TSClassification": TSClassification,
        "TSStandardize": TSStandardize,
        "accuracy": accuracy,
        "arch_obj": getattr(tsai_all, arch_name),
    }


def extract_tsai_pred_indices(pred_decoded) -> np.ndarray:
    if hasattr(pred_decoded, "detach"):
        pred = pred_decoded.detach().cpu().numpy()
    else:
        pred = np.asarray(pred_decoded)

    pred = np.asarray(pred).reshape(-1)
    return np.asarray([int(x.item() if hasattr(x, "item") else x) for x in pred], dtype=np.int64)


def run_tsai_experiment(
    exp: Dict,
    modality_info: Dict,
    base_paths: Dict[str, Path],
    config: Dict,
    tsai_obj: Dict,
) -> Tuple[Dict, pd.DataFrame]:
    torch = tsai_obj["torch"]
    get_ts_dls = tsai_obj["get_ts_dls"]
    ts_learner = tsai_obj["ts_learner"]
    TSClassification = tsai_obj["TSClassification"]
    TSStandardize = tsai_obj["TSStandardize"]
    accuracy = tsai_obj["accuracy"]
    arch_obj = tsai_obj["arch_obj"]

    X_train = to_channels_first(exp["train_X"])
    X_val = to_channels_first(exp["val_X"])
    X_test = to_channels_first(exp["test_X"])

    y_train_idx = labels_to_indices(exp["train_y"], config["activity_ids"])
    y_val_idx = labels_to_indices(exp["val_y"], config["activity_ids"])
    y_test_idx = labels_to_indices(exp["test_y"], config["activity_ids"])

    X_trainval = np.concatenate([X_train, X_val], axis=0).astype(np.float32)
    y_trainval = np.concatenate([y_train_idx, y_val_idx], axis=0).astype(np.int64)

    train_idx = np.arange(0, len(X_train), dtype=np.int64)
    val_idx = np.arange(len(X_train), len(X_trainval), dtype=np.int64)
    splits = (train_idx, val_idx)

    print("\n" + "=" * 90)
    print(f"[tsai] {exp['name']} | modality={exp['modality']}")
    print("=" * 90)
    print("Architecture:", config["tsai_arch"])
    print("Train:", X_train.shape, y_train_idx.shape)
    print("Val:  ", X_val.shape, y_val_idx.shape)
    print("Test: ", X_test.shape, y_test_idx.shape)

    tfms = [None, TSClassification()]
    batch_tfms = TSStandardize()

    dls = get_ts_dls(
        X_trainval,
        y_trainval,
        splits=splits,
        tfms=tfms,
        batch_tfms=batch_tfms,
        bs=int(config["tsai_batch_size"]),
    )

    learn = ts_learner(
        dls,
        arch_obj,
        metrics=accuracy,
    )

    learn.fit_one_cycle(int(config["tsai_epochs"]), float(config["tsai_lr"]))

    pred_output = learn.get_X_preds(
        X_test,
        y_test_idx,
        bs=int(config["tsai_batch_size"]),
        with_decoded=True,
    )

    pred_idx = extract_tsai_pred_indices(pred_output[-1])

    y_true = indices_to_labels(y_test_idx, config["activity_ids"])
    y_pred = indices_to_labels(pred_idx, config["activity_ids"])

    metrics = compute_classification_metrics(y_true, y_pred, config["activity_ids"])

    activity_df = save_run_outputs(
        y_true=y_true,
        y_pred=y_pred,
        framework="tsai",
        model_name=str(config["tsai_arch"]),
        experiment_name=exp["name"],
        synthetic_method=exp["synthetic_method"],
        modality=exp["modality"],
        modality_info=modality_info,
        base_paths=base_paths,
        config=config,
    )

    if bool(config["save_models"]):
        try:
            run_dirs = get_run_dirs(base_paths, "tsai", exp["name"], exp["modality"])
            make_dirs(run_dirs["models_dir"])
            model_path = run_dirs["models_dir"] / f"{run_dirs['run_key']}.pkl"
            learn.export(model_path)
            print("Saved:", model_path)
        except Exception as exc:
            warnings.warn(f"Could not save tsai model: {repr(exc)}")

    row = {
        "framework": "tsai",
        "model": str(config["tsai_arch"]),
        "experiment": exp["name"],
        "synthetic_method": exp["synthetic_method"],
        "synthetic_method_display_name": (
            "none" if exp["synthetic_method"] == "none"
            else method_display_name(exp["synthetic_method"], config)
        ),
        "modality": exp["modality"],
        "native_hz": int(modality_info["native_hz"]),
        "channels": ",".join(modality_info["channel_names"]),
        "description": modality_info["description"],
        "train_shape_native_N_T_C": list(exp["train_X"].shape),
        "test_shape_native_N_T_C": list(exp["test_X"].shape),
        "train_shape_framework_N_C_T": list(X_train.shape),
        "test_shape_framework_N_C_T": list(X_test.shape),
        "train_windows": int(len(exp["train_y"])),
        "test_windows": int(len(exp["test_y"])),
        "tsai_arch": str(config["tsai_arch"]),
        "tsai_epochs": int(config["tsai_epochs"]),
        "tsai_batch_size": int(config["tsai_batch_size"]),
        "tsai_lr": float(config["tsai_lr"]),
        **metrics,
    }

    try:
        del learn, dls
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

    print(json.dumps(row, indent=2))

    return row, activity_df


# ============================================================
# Runner
# ============================================================

def run_framework_experiments(
    experiments_by_modality: Dict[str, List[Dict]],
    base_paths: Dict[str, Path],
    config: Dict,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    result_rows = []
    activity_tables = []

    tsai_obj = None

    if "tsai" in config["frameworks"]:
        try:
            tsai_obj = import_tsai_stuff(config)
        except Exception as exc:
            warnings.warn(f"tsai unavailable; skipping tsai. Reason: {repr(exc)}")
            tsai_obj = None

    aeon_available = "aeon" in config["frameworks"]

    for modality, experiments in experiments_by_modality.items():
        modality_info = get_modality_info(modality, config)

        for exp in experiments:
            if aeon_available:
                try:
                    row, activity_df = run_aeon_experiment(
                        exp=exp,
                        modality_info=modality_info,
                        base_paths=base_paths,
                        config=config,
                    )
                    result_rows.append(row)
                    activity_tables.append(activity_df)
                except Exception as exc:
                    warnings.warn(f"[aeon] failed {exp['name']} | {modality}: {repr(exc)}")
                    result_rows.append(
                        {
                            "framework": "aeon",
                            "experiment": exp["name"],
                            "synthetic_method": exp["synthetic_method"],
                            "modality": modality,
                            "error": repr(exc),
                        }
                    )

            if tsai_obj is not None:
                try:
                    row, activity_df = run_tsai_experiment(
                        exp=exp,
                        modality_info=modality_info,
                        base_paths=base_paths,
                        config=config,
                        tsai_obj=tsai_obj,
                    )
                    result_rows.append(row)
                    activity_tables.append(activity_df)
                except Exception as exc:
                    warnings.warn(f"[tsai] failed {exp['name']} | {modality}: {repr(exc)}")
                    result_rows.append(
                        {
                            "framework": "tsai",
                            "model": str(config["tsai_arch"]),
                            "experiment": exp["name"],
                            "synthetic_method": exp["synthetic_method"],
                            "modality": modality,
                            "error": repr(exc),
                        }
                    )

    results_df = pd.DataFrame(result_rows)

    if len(activity_tables) > 0:
        activity_df = pd.concat(activity_tables, ignore_index=True)
    else:
        activity_df = pd.DataFrame()

    return results_df, activity_df


def make_summary_tables(results_df: pd.DataFrame, activity_df: pd.DataFrame, base_paths: Dict[str, Path]) -> Dict[str, Path]:
    paths = {}

    aggregate_path = base_paths["results_model_dir"] / "aggregate_downstream_results.csv"
    results_df.to_csv(aggregate_path, index=False)
    paths["aggregate_downstream_results"] = aggregate_path
    print("Saved:", aggregate_path)

    activity_path = base_paths["results_model_dir"] / "per_activity_downstream_results.csv"
    activity_df.to_csv(activity_path, index=False)
    paths["per_activity_downstream_results"] = activity_path
    print("Saved:", activity_path)

    if len(results_df) > 0 and "macro_f1" in results_df.columns:
        valid = results_df.dropna(subset=["macro_f1"]).copy()

        if len(valid) > 0:
            ranking = valid.sort_values(
                by=["framework", "modality", "macro_f1"],
                ascending=[True, True, False],
            )
            ranking_path = base_paths["results_model_dir"] / "downstream_macro_f1_ranking.csv"
            ranking.to_csv(ranking_path, index=False)
            paths["downstream_macro_f1_ranking"] = ranking_path
            print("Saved:", ranking_path)

            comparison_cols = [
                "framework",
                "modality",
                "experiment",
                "synthetic_method",
                "synthetic_method_display_name",
                "accuracy",
                "macro_f1",
                "weighted_f1",
                "balanced_accuracy",
            ]
            available_cols = [c for c in comparison_cols if c in valid.columns]
            compact = valid[available_cols].copy()
            compact_path = base_paths["results_model_dir"] / "compact_downstream_comparison.csv"
            compact.to_csv(compact_path, index=False)
            paths["compact_downstream_comparison"] = compact_path
            print("Saved:", compact_path)

    return paths


def main(config: Dict = EVAL_CONFIG) -> Dict[str, object]:
    set_random_seeds(int(config["random_seed"]))

    base_paths = get_base_paths(config)

    make_dirs(
        base_paths["results_model_dir"],
        base_paths["figures_model_dir"],
        base_paths["models_model_dir"],
        base_paths["configs_dir"],
    )

    config_path = base_paths["configs_dir"] / f"downstream_activity_eval_{config['model_family']}_config.json"
    save_json(config, config_path)

    print("=" * 100)
    print("Notebook 06: Downstream activity detection evaluation")
    print("=" * 100)
    print("Model family:", config["model_family"])
    print("Frameworks:", config["frameworks"])
    print("Modalities:", config["modalities"])
    print("Synthetic methods:", config["synthetic_methods"])
    print("Real dir:", base_paths["real_dir"])
    print("Synthetic base:", base_paths["synthetic_base_dir"])
    print("Results dir:", base_paths["results_model_dir"])
    print("Figures dir:", base_paths["figures_model_dir"])
    print("Models dir:", base_paths["models_model_dir"])
    print("Config saved:", config_path)

    print("\nLoading real data...")
    real_X_all, real_y_all, real_subjects_all = load_real_native_data(base_paths["real_dir"], config)

    real_splits = prepare_real_splits(
        real_X_all=real_X_all,
        real_y_all=real_y_all,
        real_subjects_all=real_subjects_all,
        config=config,
    )

    print("\nReal split sizes:")
    for split_name, (_, y_split, subjects_split) in real_splits.items():
        print(
            f"  {split_name:>5}: windows={len(y_split):6d} | "
            f"subjects={sorted(np.unique(subjects_split.astype(str)), key=subject_sort_key)}"
        )

    real_train_X_raw, real_train_y, real_train_subjects = real_splits["train"]
    real_val_X_raw, real_val_y, real_val_subjects = real_splits["val"]
    real_test_X_raw, real_test_y, real_test_subjects = real_splits["test"]

    print("\nBuilding real modality views...")
    real_train_views = build_eval_raw_views(real_train_X_raw, config)
    real_val_views = build_eval_raw_views(real_val_X_raw, config)
    real_test_views = build_eval_raw_views(real_test_X_raw, config)

    synthetic_views_by_method = {}
    synthetic_y_by_method = {}
    synthetic_subjects_by_method = {}

    synthetic_test_views_by_method = {}
    synthetic_test_y_by_method = {}
    synthetic_test_subjects_by_method = {}

    for method_name in config["synthetic_methods"]:
        synthetic_dir = base_paths["synthetic_base_dir"] / method_name

        print("\n" + "#" * 100)
        print(f"Loading synthetic method: {method_name} ({method_display_name(method_name, config)})")
        print("#" * 100)

        syn_X_raw, syn_y, syn_subjects = load_synthetic_native_data(synthetic_dir, config)
        syn_views = build_eval_raw_views(syn_X_raw, config)

        synthetic_views_by_method[method_name] = syn_views
        synthetic_y_by_method[method_name] = syn_y
        synthetic_subjects_by_method[method_name] = syn_subjects

        selected_syn_test_subjects = select_synthetic_test_subjects(syn_subjects, config)
        print("Selected synthetic test subjects:", selected_syn_test_subjects)

        syn_test_X_raw, syn_test_y, syn_test_subjects = filter_by_subjects(
            X_dict=syn_X_raw,
            y=syn_y,
            subjects=syn_subjects,
            selected_subjects=selected_syn_test_subjects,
        )
        syn_test_views = build_eval_raw_views(syn_test_X_raw, config)

        synthetic_test_views_by_method[method_name] = syn_test_views
        synthetic_test_y_by_method[method_name] = syn_test_y
        synthetic_test_subjects_by_method[method_name] = syn_test_subjects

    save_split_summary(
        real_splits=real_splits,
        synthetic_summaries={
            method: (synthetic_y_by_method[method], synthetic_subjects_by_method[method])
            for method in config["synthetic_methods"]
        },
        synthetic_test_summaries={
            method: (synthetic_test_y_by_method[method], synthetic_test_subjects_by_method[method])
            for method in config["synthetic_methods"]
        },
        base_paths=base_paths,
        config=config,
    )

    experiments_by_modality = {}

    for modality in config["modalities"]:
        experiments_by_modality[modality] = build_experiments_for_modality(
            modality=modality,
            real_train_views=real_train_views,
            real_val_views=real_val_views,
            real_test_views=real_test_views,
            real_train_y=real_train_y,
            real_val_y=real_val_y,
            real_test_y=real_test_y,
            synthetic_views_by_method=synthetic_views_by_method,
            synthetic_y_by_method=synthetic_y_by_method,
            synthetic_test_views_by_method=synthetic_test_views_by_method,
            synthetic_test_y_by_method=synthetic_test_y_by_method,
            config=config,
        )

    shape_report = save_shape_report(experiments_by_modality, base_paths)

    print("\nExperiment shape report:")
    safe_display(shape_report)

    results_df, activity_df = run_framework_experiments(
        experiments_by_modality=experiments_by_modality,
        base_paths=base_paths,
        config=config,
    )

    output_paths = make_summary_tables(results_df, activity_df, base_paths)

    summary = {
        "model_family": config["model_family"],
        "frameworks": config["frameworks"],
        "modalities": config["modalities"],
        "synthetic_methods": config["synthetic_methods"],
        "activity_ids": config["activity_ids"],
        "results_model_dir": str(base_paths["results_model_dir"]),
        "figures_model_dir": str(base_paths["figures_model_dir"]),
        "models_model_dir": str(base_paths["models_model_dir"]),
        "config_path": str(config_path),
        "output_paths": {key: str(value) for key, value in output_paths.items()},
        "main_question": "Does training with synthetic KoVAE data improve activity recognition on real held-out test subjects?",
        "evaluation_design": [
            "train 10 real subjects -> test 3 real subjects",
            "train 10 real subjects -> test 3 synthetic subjects",
            "train 10 synthetic subjects -> test 3 real subjects",
            "train 10 real + 10 synthetic subjects -> test 3 real subjects"
        ],
        "synthetic_test_subject_selection": config["synthetic_test_subject_selection"],
        "num_synthetic_test_subjects": int(config["num_synthetic_test_subjects"]),
        "most_important_comparison": "real_to_real vs real_plus_synthetic_to_real__posterior_bank_v2",
    }

    summary_path = base_paths["results_model_dir"] / "downstream_evaluation_summary.json"
    save_json(summary, summary_path)
    output_paths["downstream_evaluation_summary"] = summary_path
    print("Saved:", summary_path)

    print("\n" + "=" * 100)
    print("Downstream evaluation completed.")
    print("=" * 100)

    print("\nMain outputs:")
    for key, value in output_paths.items():
        print(f"  {key}: {value}")

    print("\nAggregate results:")
    safe_display(results_df)

    print("\nPer-activity results:")
    safe_display(activity_df)

    return {
        "base_paths": base_paths,
        "shape_report": shape_report,
        "results_df": results_df,
        "activity_df": activity_df,
        "output_paths": output_paths,
    }


if __name__ == "__main__":
    outputs = main(EVAL_CONFIG)


Notebook 06: Downstream activity detection evaluation
Model family: kovae
Frameworks: ['aeon']
Modalities: ['fused']
Synthetic methods: ['rollout_v1', 'posterior_bank_v2']
Real dir: /home/iailab42/khans1/projects/ir/data/processed/native_rates
Synthetic base: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae
Results dir: /home/iailab42/khans1/projects/ir/results/downstream/kovae
Figures dir: /home/iailab42/khans1/projects/ir/figures/downstream/kovae
Models dir: /home/iailab42/khans1/projects/ir/models/downstream/kovae
Config saved: /home/iailab42/khans1/projects/ir/configs/downstream_activity_eval_kovae_config.json

Loading real data...
Loaded real acc: (46925, 256, 3)
Loaded real bvp: (46925, 512, 1)
Loaded real slow: (46925, 32, 2)
Subject split OK.

Real split sizes:
  train: windows= 30762 | subjects=[np.str_('S1'), np.str_('S2'), np.str_('S3'), np.str_('S4'), np.str_('S5'), np.str_('S6'), np.str_('S9'), np.str_('S11'), np.str_('S12'), np.str_('S13')]
    val: windows

,experiment,synthetic_method,modality,train_shape_native_N_T_C,val_shape_native_N_T_C,test_shape_native_N_T_C,train_windows,val_windows,test_windows
0,real_to_real,none,fused,"[30762, 512, 6]","[6100, 512, 6]","[10063, 512, 6]",30762,6100,10063
1,synthetic_to_real__rollout_v1,rollout_v1,fused,"[30000, 512, 6]","[6100, 512, 6]","[10063, 512, 6]",30000,6100,10063
2,real_plus_synthetic_to_real__rollout_v1,rollout_v1,fused,"[60762, 512, 6]","[6100, 512, 6]","[10063, 512, 6]",60762,6100,10063
3,real_to_3synthetic__rollout_v1,rollout_v1,fused,"[30762, 512, 6]","[6100, 512, 6]","[9000, 512, 6]",30762,6100,9000
4,synthetic_to_real__posterior_bank_v2,posterior_bank_v2,fused,"[30000, 512, 6]","[6100, 512, 6]","[10063, 512, 6]",30000,6100,10063
5,real_plus_synthetic_to_real__posterior_bank_v2,posterior_bank_v2,fused,"[60762, 512, 6]","[6100, 512, 6]","[10063, 512, 6]",60762,6100,10063
6,real_to_3synthetic__posterior_bank_v2,posterior_bank_v2,fused,"[30762, 512, 6]","[6100, 512, 6]","[9000, 512, 6]",30762,6100,9000


aeon low-variation filter: dropped 661 of 30762 windows where at least one channel had std <= 1e-06
aeon low-variation filter: dropped 125 of 10063 windows where at least one channel had std <= 1e-06

[aeon] real_to_real | modality=fused
Model: MiniRocketClassifier
Train: (30101, 6, 512) (30101,)
Test:  (9938, 6, 512) (9938,)


/home/iai/user/khans1/.local/lib/python3.12/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


Saved: /home/iailab42/khans1/projects/ir/results/downstream/kovae/predictions/aeon__real_to_real__fused_predictions.csv
Saved: /home/iailab42/khans1/projects/ir/results/downstream/kovae/confusion_matrices/aeon__real_to_real__fused_confusion_matrix.csv
Saved: /home/iailab42/khans1/projects/ir/figures/downstream/kovae/confusion_matrices/aeon__real_to_real__fused_confusion_matrix.png
Saved: /home/iailab42/khans1/projects/ir/results/downstream/kovae/per_activity_reports/aeon__real_to_real__fused_per_activity_metrics.csv
Saved: /home/iailab42/khans1/projects/ir/models/downstream/kovae/aeon/aeon__real_to_real__fused.joblib
{
  "framework": "aeon",
  "model": "MiniRocketClassifier",
  "experiment": "real_to_real",
  "synthetic_method": "none",
  "synthetic_method_display_name": "none",
  "modality": "fused",
  "native_hz": 64,
  "channels": "ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",
  "description": "Fused ACC+BVP+EDA+TEMP resampled to length 512",
  "train_shape_native_N_T_C": [
    30762,
    512,
 

,framework,model,experiment,synthetic_method,synthetic_method_display_name,modality,native_hz,channels,description,train_shape_native_N_T_C,...,aeon_min_std,aeon_ramp_scale,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,balanced_accuracy
0,aeon,MiniRocketClassifier,real_to_real,none,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30762, 512, 6]",...,0.000001,0.001,0.677199,0.721391,0.770520,0.727710,0.704578,0.677199,0.674676,0.770520
1,aeon,MiniRocketClassifier,synthetic_to_real__rollout_v1,rollout_v1,KoVAE-Rollout,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30000, 512, 6]",...,0.000001,0.001,0.302576,0.417091,0.447039,0.295033,0.491480,0.302576,0.213106,0.447039
2,aeon,MiniRocketClassifier,real_plus_synthetic_to_real__rollout_v1,rollout_v1,KoVAE-Rollout,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[60762, 512, 6]",...,0.000001,0.001,0.700443,0.727093,0.767803,0.728298,0.725895,0.700443,0.702437,0.767803
3,aeon,MiniRocketClassifier,real_to_3synthetic__rollout_v1,rollout_v1,KoVAE-Rollout,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30762, 512, 6]",...,0.000001,0.001,0.292333,0.110562,0.137660,0.077326,0.174353,0.292333,0.150358,0.137660
4,aeon,MiniRocketClassifier,synthetic_to_real__posterior_bank_v2,posterior_bank_v2,KoVAE-Posterior,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30000, 512, 6]",...,0.000001,0.001,0.650634,0.686141,0.726573,0.668809,0.683286,0.650634,0.629859,0.726573
5,aeon,MiniRocketClassifier,real_plus_synthetic_to_real__posterior_bank_v2,posterior_bank_v2,KoVAE-Posterior,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[60762, 512, 6]",...,0.000001,0.001,0.707587,0.736364,0.782691,0.736641,0.742741,0.707587,0.712314,0.782691
6,aeon,MiniRocketClassifier,real_to_3synthetic__posterior_bank_v2,posterior_bank_v2,KoVAE-Posterior,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30762, 512, 6]",...,0.000001,0.001,0.510111,0.852799,0.447283,0.491619,0.743135,0.510111,0.448417,0.447283



Per-activity results:


,framework,model,experiment,synthetic_method,modality,native_hz,channels,activity_label,precision,recall,f1,support,correct_true_activity_windows,total_true_activity_windows,true_activity_window_accuracy
0,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",1,0.518874,0.937965,0.668140,806,756,806,0.937965
1,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",2,0.899007,0.855118,0.876513,635,543,635,0.855118
2,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",3,0.498018,0.849099,0.627810,444,377,444,0.849099
3,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",4,1.000000,0.969871,0.984705,697,676,697,0.969871
4,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",5,0.895785,0.858297,0.876640,1362,1169,1362,0.858297
5,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",6,0.674905,0.453674,0.542606,3130,1420,3130,0.453674
6,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",7,0.767389,0.598754,0.672664,1124,673,1124,0.598754
7,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",8,0.517146,0.641379,0.572601,1740,1116,1740,0.641379
8,aeon,MiniRocketClassifier,synthetic_to_real__rollout_v1,rollout_v1,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",1,0.208263,0.913151,0.339171,806,736,806,0.913151
9,aeon,MiniRocketClassifier,synthetic_to_real__rollout_v1,rollout_v1,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",2,0.653310,0.590551,0.620347,635,375,635,0.590551


## Final run

This version fixes the aeon error:

```text
Input collection has too little variation
```

It does this by dropping only the near-flat windows from the temporary aeon arrays.

The output CSV records how many windows were dropped:

```text
train_windows_dropped_by_aeon_filter
test_windows_dropped_by_aeon_filter
```


In [2]:
EVAL_CONFIG["project_root"] = "/home/iailab42/khans1/projects/ir"
EVAL_CONFIG["model_family"] = "kovae"

EVAL_CONFIG["real_dir"] = "data/processed/native_rates"
EVAL_CONFIG["synthetic_base_dir"] = "data/synthetic_subjects/kovae"

EVAL_CONFIG["synthetic_methods"] = ["rollout_v1", "posterior_bank_v2"]

EVAL_CONFIG["method_display_names"] = {
    "rollout_v1": "KoVAE-Rollout",
    "posterior_bank_v2": "KoVAE-Posterior",
}

# Exact requested evaluation design.
EVAL_CONFIG["include_real_to_synthetic"] = True
EVAL_CONFIG["synthetic_test_subject_selection"] = "first_n"
EVAL_CONFIG["num_synthetic_test_subjects"] = 3
EVAL_CONFIG["specific_synthetic_test_subjects"] = []

# aeon evaluation.
EVAL_CONFIG["frameworks"] = ["aeon"]
EVAL_CONFIG["modalities"] = ["fused"]

# Fix MiniRocket/Rocket error for nearly constant case/channel pairs.
# Safer than ramp because it guarantees aeon will not see flat windows.
EVAL_CONFIG["aeon_fix_low_variation"] = True
EVAL_CONFIG["aeon_low_variation_strategy"] = "drop_cases"
EVAL_CONFIG["aeon_min_std"] = 1e-6
EVAL_CONFIG["aeon_ramp_scale"] = 1e-3

# Use all training windows by default.
EVAL_CONFIG["max_train_windows_per_experiment"] = None

EVAL_CONFIG["aeon_n_kernels"] = 5000
EVAL_CONFIG["aeon_n_jobs"] = -1

EVAL_CONFIG["save_models"] = True
EVAL_CONFIG["save_confusion_matrix_figures"] = True
EVAL_CONFIG["show_confusion_matrix_figures"] = False

outputs = main(EVAL_CONFIG)
outputs["results_df"]


Notebook 06: Downstream activity detection evaluation
Model family: kovae
Frameworks: ['aeon']
Modalities: ['fused']
Synthetic methods: ['rollout_v1', 'posterior_bank_v2']
Real dir: /home/iailab42/khans1/projects/ir/data/processed/native_rates
Synthetic base: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae
Results dir: /home/iailab42/khans1/projects/ir/results/downstream/kovae
Figures dir: /home/iailab42/khans1/projects/ir/figures/downstream/kovae
Models dir: /home/iailab42/khans1/projects/ir/models/downstream/kovae
Config saved: /home/iailab42/khans1/projects/ir/configs/downstream_activity_eval_kovae_config.json

Loading real data...
Loaded real acc: (46925, 256, 3)
Loaded real bvp: (46925, 512, 1)
Loaded real slow: (46925, 32, 2)
Subject split OK.

Real split sizes:
  train: windows= 30762 | subjects=[np.str_('S1'), np.str_('S2'), np.str_('S3'), np.str_('S4'), np.str_('S5'), np.str_('S6'), np.str_('S9'), np.str_('S11'), np.str_('S12'), np.str_('S13')]
    val: windows

,experiment,synthetic_method,modality,train_shape_native_N_T_C,val_shape_native_N_T_C,test_shape_native_N_T_C,train_windows,val_windows,test_windows
0,real_to_real,none,fused,"[30762, 512, 6]","[6100, 512, 6]","[10063, 512, 6]",30762,6100,10063
1,synthetic_to_real__rollout_v1,rollout_v1,fused,"[30000, 512, 6]","[6100, 512, 6]","[10063, 512, 6]",30000,6100,10063
2,real_plus_synthetic_to_real__rollout_v1,rollout_v1,fused,"[60762, 512, 6]","[6100, 512, 6]","[10063, 512, 6]",60762,6100,10063
3,real_to_3synthetic__rollout_v1,rollout_v1,fused,"[30762, 512, 6]","[6100, 512, 6]","[9000, 512, 6]",30762,6100,9000
4,synthetic_to_real__posterior_bank_v2,posterior_bank_v2,fused,"[30000, 512, 6]","[6100, 512, 6]","[10063, 512, 6]",30000,6100,10063
5,real_plus_synthetic_to_real__posterior_bank_v2,posterior_bank_v2,fused,"[60762, 512, 6]","[6100, 512, 6]","[10063, 512, 6]",60762,6100,10063
6,real_to_3synthetic__posterior_bank_v2,posterior_bank_v2,fused,"[30762, 512, 6]","[6100, 512, 6]","[9000, 512, 6]",30762,6100,9000


aeon low-variation filter: dropped 661 of 30762 windows where at least one channel had std <= 1e-06
aeon low-variation filter: dropped 125 of 10063 windows where at least one channel had std <= 1e-06

[aeon] real_to_real | modality=fused
Model: MiniRocketClassifier
Train: (30101, 6, 512) (30101,)
Test:  (9938, 6, 512) (9938,)
Saved: /home/iailab42/khans1/projects/ir/results/downstream/kovae/predictions/aeon__real_to_real__fused_predictions.csv
Saved: /home/iailab42/khans1/projects/ir/results/downstream/kovae/confusion_matrices/aeon__real_to_real__fused_confusion_matrix.csv
Saved: /home/iailab42/khans1/projects/ir/figures/downstream/kovae/confusion_matrices/aeon__real_to_real__fused_confusion_matrix.png
Saved: /home/iailab42/khans1/projects/ir/results/downstream/kovae/per_activity_reports/aeon__real_to_real__fused_per_activity_metrics.csv
Saved: /home/iailab42/khans1/projects/ir/models/downstream/kovae/aeon/aeon__real_to_real__fused.joblib
{
  "framework": "aeon",
  "model": "MiniRocket

,framework,model,experiment,synthetic_method,synthetic_method_display_name,modality,native_hz,channels,description,train_shape_native_N_T_C,...,aeon_min_std,aeon_ramp_scale,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,balanced_accuracy
0,aeon,MiniRocketClassifier,real_to_real,none,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30762, 512, 6]",...,0.000001,0.001,0.677199,0.721391,0.770520,0.727710,0.704578,0.677199,0.674676,0.770520
1,aeon,MiniRocketClassifier,synthetic_to_real__rollout_v1,rollout_v1,KoVAE-Rollout,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30000, 512, 6]",...,0.000001,0.001,0.302576,0.417091,0.447039,0.295033,0.491480,0.302576,0.213106,0.447039
2,aeon,MiniRocketClassifier,real_plus_synthetic_to_real__rollout_v1,rollout_v1,KoVAE-Rollout,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[60762, 512, 6]",...,0.000001,0.001,0.700443,0.727093,0.767803,0.728298,0.725895,0.700443,0.702437,0.767803
3,aeon,MiniRocketClassifier,real_to_3synthetic__rollout_v1,rollout_v1,KoVAE-Rollout,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30762, 512, 6]",...,0.000001,0.001,0.292333,0.110562,0.137660,0.077326,0.174353,0.292333,0.150358,0.137660
4,aeon,MiniRocketClassifier,synthetic_to_real__posterior_bank_v2,posterior_bank_v2,KoVAE-Posterior,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30000, 512, 6]",...,0.000001,0.001,0.650634,0.686141,0.726573,0.668809,0.683286,0.650634,0.629859,0.726573
5,aeon,MiniRocketClassifier,real_plus_synthetic_to_real__posterior_bank_v2,posterior_bank_v2,KoVAE-Posterior,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[60762, 512, 6]",...,0.000001,0.001,0.707587,0.736364,0.782691,0.736641,0.742741,0.707587,0.712314,0.782691
6,aeon,MiniRocketClassifier,real_to_3synthetic__posterior_bank_v2,posterior_bank_v2,KoVAE-Posterior,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30762, 512, 6]",...,0.000001,0.001,0.510111,0.852799,0.447283,0.491619,0.743135,0.510111,0.448417,0.447283



Per-activity results:


,framework,model,experiment,synthetic_method,modality,native_hz,channels,activity_label,precision,recall,f1,support,correct_true_activity_windows,total_true_activity_windows,true_activity_window_accuracy
0,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",1,0.518874,0.937965,0.668140,806,756,806,0.937965
1,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",2,0.899007,0.855118,0.876513,635,543,635,0.855118
2,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",3,0.498018,0.849099,0.627810,444,377,444,0.849099
3,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",4,1.000000,0.969871,0.984705,697,676,697,0.969871
4,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",5,0.895785,0.858297,0.876640,1362,1169,1362,0.858297
5,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",6,0.674905,0.453674,0.542606,3130,1420,3130,0.453674
6,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",7,0.767389,0.598754,0.672664,1124,673,1124,0.598754
7,aeon,MiniRocketClassifier,real_to_real,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",8,0.517146,0.641379,0.572601,1740,1116,1740,0.641379
8,aeon,MiniRocketClassifier,synthetic_to_real__rollout_v1,rollout_v1,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",1,0.208263,0.913151,0.339171,806,736,806,0.913151
9,aeon,MiniRocketClassifier,synthetic_to_real__rollout_v1,rollout_v1,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",2,0.653310,0.590551,0.620347,635,375,635,0.590551


,framework,model,experiment,synthetic_method,synthetic_method_display_name,modality,native_hz,channels,description,train_shape_native_N_T_C,...,aeon_min_std,aeon_ramp_scale,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,balanced_accuracy
0,aeon,MiniRocketClassifier,real_to_real,none,none,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30762, 512, 6]",...,0.000001,0.001,0.677199,0.721391,0.770520,0.727710,0.704578,0.677199,0.674676,0.770520
1,aeon,MiniRocketClassifier,synthetic_to_real__rollout_v1,rollout_v1,KoVAE-Rollout,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30000, 512, 6]",...,0.000001,0.001,0.302576,0.417091,0.447039,0.295033,0.491480,0.302576,0.213106,0.447039
2,aeon,MiniRocketClassifier,real_plus_synthetic_to_real__rollout_v1,rollout_v1,KoVAE-Rollout,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[60762, 512, 6]",...,0.000001,0.001,0.700443,0.727093,0.767803,0.728298,0.725895,0.700443,0.702437,0.767803
3,aeon,MiniRocketClassifier,real_to_3synthetic__rollout_v1,rollout_v1,KoVAE-Rollout,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30762, 512, 6]",...,0.000001,0.001,0.292333,0.110562,0.137660,0.077326,0.174353,0.292333,0.150358,0.137660
4,aeon,MiniRocketClassifier,synthetic_to_real__posterior_bank_v2,posterior_bank_v2,KoVAE-Posterior,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30000, 512, 6]",...,0.000001,0.001,0.650634,0.686141,0.726573,0.668809,0.683286,0.650634,0.629859,0.726573
5,aeon,MiniRocketClassifier,real_plus_synthetic_to_real__posterior_bank_v2,posterior_bank_v2,KoVAE-Posterior,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[60762, 512, 6]",...,0.000001,0.001,0.707587,0.736364,0.782691,0.736641,0.742741,0.707587,0.712314,0.782691
6,aeon,MiniRocketClassifier,real_to_3synthetic__posterior_bank_v2,posterior_bank_v2,KoVAE-Posterior,fused,64,"ACC_x,ACC_y,ACC_z,BVP,EDA,TEMP",Fused ACC+BVP+EDA+TEMP resampled to length 512,"[30762, 512, 6]",...,0.000001,0.001,0.510111,0.852799,0.447283,0.491619,0.743135,0.510111,0.448417,0.447283
